#`Setup And Update ComfyUI`



In [ ]:
from pathlib import Path

OPTIONS = {}

DRIVE_PATH = "/content/drive/MyDrive"  # @param {type:"string"}
UPDATE_COMFY_UI = True  #@param {type:"boolean"}
WORKSPACE = '/content/ComfyUI'
MODELS_PATH = Path(DRIVE_PATH) / 'ComfyUI' / 'models'
OUTPUT_PATH = Path(DRIVE_PATH) / 'vidio'
OPTIONS['UPDATE_COMFY_UI'] = UPDATE_COMFY_UI

if not DRIVE_PATH:
  raise ValueError('DRIVE_PATH is required so existing models remain available.')
if not Path(DRIVE_PATH).is_dir():
  raise RuntimeError(f'Google Drive is not mounted: {DRIVE_PATH}. Run the mount cell first.')
MODELS_PATH.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
%cd /content

if not Path(WORKSPACE).is_dir():
  !echo -= Initial setup ComfyUI =-
  !git clone https://github.com/comfyanonymous/ComfyUI {WORKSPACE}

# Keep all ComfyUI files on /content, but use the persistent Drive model directory.
local_models_path = Path(WORKSPACE) / 'models'
if local_models_path.is_symlink():
  if local_models_path.resolve() != MODELS_PATH.resolve():
    raise RuntimeError(f'{local_models_path} points to an unexpected location; update it manually.')
elif local_models_path.exists():
  local_models_backup = Path('/content/ComfyUI-local-models-backup')
  if local_models_backup.exists():
    raise RuntimeError(f'Cannot replace {local_models_path}: backup already exists at {local_models_backup}.')
  local_models_path.rename(local_models_backup)
  local_models_path.symlink_to(MODELS_PATH, target_is_directory=True)
else:
  local_models_path.symlink_to(MODELS_PATH, target_is_directory=True)

local_output_path = Path(WORKSPACE) / 'output'
if local_output_path.is_symlink():
  if local_output_path.resolve() != OUTPUT_PATH.resolve():
    raise RuntimeError(f'{local_output_path} points to an unexpected location; update it manually.')
elif local_output_path.exists():
  local_output_backup = Path('/content/ComfyUI-local-output-backup')
  if local_output_backup.exists():
    raise RuntimeError(f'Cannot replace {local_output_path}: backup already exists at {local_output_backup}.')
  local_output_path.rename(local_output_backup)
  local_output_path.symlink_to(OUTPUT_PATH, target_is_directory=True)
else:
  local_output_path.symlink_to(OUTPUT_PATH, target_is_directory=True)
print(f'ComfyUI: {WORKSPACE}')
print(f'Models: {MODELS_PATH}')
print(f'Output: {OUTPUT_PATH}')
%cd $WORKSPACE

if OPTIONS['UPDATE_COMFY_UI']:
  !echo -= Updating ComfyUI =-
  !git pull

!echo -= Install dependencies =-
!pip install xformers!=0.0.18 -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu121 --extra-index-url https://download.pytorch.org/whl/cu118 --extra-index-url https://download.pytorch.org/whl/cu117
!pip install sageattention

!echo -= Install ComfyUI-MiniMax-H3-Turbo custom node =-
CUSTOM_NODES_PATH = Path(WORKSPACE) / 'custom_nodes'
MINIMAX_H3_PATH = CUSTOM_NODES_PATH / 'ComfyUI-MiniMax-H3-Turbo'
%cd {CUSTOM_NODES_PATH}
if not MINIMAX_H3_PATH.is_dir():
  !git clone https://github.com/larryvrh/ComfyUI-MiniMax-H3-Turbo
else:
  !git -C {MINIMAX_H3_PATH} pull --ff-only

!echo -= Install ComfyUI-KJNodes custom node =-
KJNODES_PATH = CUSTOM_NODES_PATH / 'ComfyUI-KJNodes'
if not KJNODES_PATH.is_dir():
  !git clone https://github.com/kijai/ComfyUI-KJNodes
else:
  !git -C {KJNODES_PATH} pull --ff-only
!pip install -r {KJNODES_PATH}/requirements.txt

%cd {WORKSPACE}

# `START ComfyUI & Expose Server`

## Download Prerequisits

In [ ]:
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

## CF Tunnel

In [ ]:
import subprocess
import threading
import time
import socket
import urllib.request

def iframe_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      if result == 0:
        break
      sock.close()
  print("\nComfyUI finished loading, trying to launch cloudflared (if it gets stuck here cloudflared is having issues)\n")

  p = subprocess.Popen(["cloudflared", "tunnel", "--protocol", "http2", "--url", "http://127.0.0.1:{}".format(port)], stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
  for line in p.stdout:
    l = line.decode()
    if "trycloudflare.com " in l:
      print("This is the URL to access ComfyUI:", l[l.find("http"):], end='')
    else:
      print(l, end='')


threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

!python main.py --dont-print-server --highvram --disable-async-offload --use-sage-attention